<a href="https://colab.research.google.com/github/stariane/weather-forcast/blob/main/weather.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q gradio

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import gradio as grad
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# ==========================================
# 1. FETCH DATASET & TRAIN THE MODEL
# ==========================================
print("Fetching dataset and training the model... Please wait.")

# Download the Jena Climate Dataset
zip_path = tf.keras.utils.get_file(
    origin='https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip',
    fname='jena_climate_2009_2016.csv.zip',
    extract=True
)
# Correctly construct the path to the CSV file inside the extracted directory
csv_path = os.path.join(zip_path, 'jena_climate_2009_2016.csv')
df = pd.read_csv(csv_path)

# Quick preprocessing
df['Date Time'] = pd.to_datetime(df['Date Time'], format='%d.%m.%Y %H:%M:%S')
df.set_index('Date Time', inplace=True)

daily_df = pd.DataFrame()
daily_df['temp_max_tomorrow'] = df['T (degC)'].resample('D').max().shift(-1)
daily_df['temp_mean_today'] = df['T (degC)'].resample('D').mean()
daily_df['humidity_today'] = df['rh (%)'].resample('D').mean()
daily_df['pressure_today'] = df['p (mbar)'].resample('D').mean()
daily_df.dropna(inplace=True)

# Split and Train
X = daily_df[['temp_mean_today', 'humidity_today', 'pressure_today']]
y = daily_df['temp_max_tomorrow']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
print("Model training complete!")

# ==========================================
# 2. DEFINE THE PREDICTION FUNCTION
# ==========================================
def predict_weather(mean_temp, humidity, pressure):
    """
    Takes today's metrics as input from the web interface,
    formats them, and returns the AI's prediction.
    """
    # Arrange inputs into a 2D array matching the model features
    input_data = np.array([[mean_temp, humidity, pressure]])
    prediction = model.predict(input_data)[0]

    # Return a formatted string
    return f"☀️ Predicted Tomorrow's Max Temp: {prediction:.2f}°C"

# ==========================================
# 3. BUILD THE GRADIO MOBILE-FRIENDLY UI
# ==========================================
# Design the user inputs (Sliders make it very easy to use on a touch screen)
interface = grad.Interface(
    fn=predict_weather,
    inputs=[
        grad.Slider(minimum=-20, maximum=40, value=15, label="Today's Mean Temperature (°C)"),
        grad.Slider(minimum=0, maximum=100, value=70, label="Today's Average Humidity (%)"),
        grad.Slider(minimum=900, maximum=1050, value=1013, label="Today's Atmospheric Pressure (mbar)")
    ],
    outputs=grad.Textbox(label="AI Weather Forecast"),
    title="⛅ AI Weather Predictor",
    description="Move the sliders to represent today's weather conditions, and the trained AI will forecast tomorrow's maximum temperature. Open this link on your phone!",
    theme="soft"
)

# CRITICAL: share=True generates the public link for your mobile device
interface.launch(share=True)

Fetching dataset and training the model... Please wait.
Model training complete!


/usr/local/lib/python3.13/dist-packages/gradio/interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://42e8ed5fde7492e423.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
